# Multilingual Keyword-Based Retrieval (BM25) Implementation

Author: Daria

This notebook implements and tests the baseline retrieval component for the project assignment:
- Indexes documents in their original language (EN or DE)
- Detects query language
- Translates the query to the other language
- Searches both EN and DE documents using BM25 for cross-lingual retrieval


In [26]:
!pip install deep-translator nltk rank-bm25 langdetect


[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [27]:
import os
from typing import List, Dict, Any, Tuple, Optional
import numpy as np
from pathlib import Path
import json

# For language detection
from langdetect import detect

# For translation
from deep_translator import GoogleTranslator

# For BM25 retrieval
from rank_bm25 import BM25Okapi
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

## 1. Define the Multilingual BM25 Retriever Class

In [35]:
class MultilingualBM25Retriever:
    """
    A retrieval system that:
    1. Indexes documents in their original language (EN or DE)
    2. Detects query language
    3. Translates the query to the other language
    4. Searches both EN and DE documents using BM25
    """
    
    def __init__(self, docs_directory: str = "data/documents"):
        """
        Initialize the retriever.
        
        Args:
            docs_directory: Path to the directory containing JSON documents
        """
        self.docs_directory = docs_directory
        
        # Storage for documents by language
        self.documents = {
            "en": [],
            "de": []
        }
        
        # BM25 indexes for each language
        self.bm25_indexes = {
            "en": None,
            "de": None
        }
        
        # Tokenized corpus for each language
        self.tokenized_corpus = {
            "en": [],
            "de": []
        }
        
        # Load stopwords
        self.stopwords = {
            "en": set(stopwords.words('english')),
            "de": set(stopwords.words('german'))
        }
        
        # Load documents and build indexes
        self._load_documents()
        self._build_indexes()
    
    def _load_documents(self):
        """Load JSON documents from directory and separate by language"""
        print(f"Loading documents from {self.docs_directory}...")
        
        # Get all JSON files in the directory
        json_files = [f for f in os.listdir(self.docs_directory) if f.endswith('.json')]
        
        for file_name in json_files:
            file_path = os.path.join(self.docs_directory, file_name)
            
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    doc = json.load(f)
                    
                language = doc.get("language", "").lower()
                if language in ["en", "de"]:
                    self.documents[language].append(doc)
                else:
                    print(f"Skipping document with unknown language: {language} - {file_path}")
            
            except Exception as e:
                print(f"Error loading {file_path}: {e}")
        
        print(f"Loaded {len(self.documents['en'])} English documents and {len(self.documents['de'])} German documents")
    
    def _preprocess_text(self, text: str, language: str) -> List[str]:
        """
        Preprocess text for BM25 indexing
        
        Args:
            text: Text to preprocess
            language: Language of the text ('en' or 'de')
            
        Returns:
            List of tokens
        """
        # Tokenize
        tokens = word_tokenize(text.lower())
        
        # Remove stopwords and non-alphabetic tokens
        tokens = [token for token in tokens 
                 if token.isalpha() and token not in self.stopwords.get(language, set())]
        
        return tokens
    
    def _build_indexes(self):
        """Build BM25 indexes with improved field weighting"""
        print("Building BM25 indexes...")
        
        # Process English documents
        for doc in self.documents["en"]:
            # Extract fields
            title = doc.get("title", "")
            content = doc.get("main_content", "")
            summary = doc.get("summary", "")
            keywords = doc.get("keywords", [])
            
            # Stronger field weighting
            # Repeat title and keywords more times to increase their importance
            combined_text = f"{title} {title} {title} {title} "  # 4x weight for title
            combined_text += f"{summary} {summary} "  # 2x weight for summary
            
            # Add keywords with high weight
            keyword_text = " ".join(keywords)
            combined_text += f"{keyword_text} {keyword_text} {keyword_text} "  # 3x weight for keywords
            
            # Add main content
            combined_text += content
            
            # Tokenize and add to corpus
            tokenized_doc = self._preprocess_text(combined_text, "en")
            self.tokenized_corpus["en"].append(tokenized_doc)
        
        # Process German documents (same approach)
        for doc in self.documents["de"]:
            # Extract fields
            title = doc.get("title", "")
            content = doc.get("main_content", "")
            summary = doc.get("summary", "")
            keywords = doc.get("keywords", [])
            
            # Stronger field weighting
            combined_text = f"{title} {title} {title} {title} "  # 4x weight for title
            combined_text += f"{summary} {summary} "  # 2x weight for summary
            
            # Add keywords with high weight
            keyword_text = " ".join(keywords)
            combined_text += f"{keyword_text} {keyword_text} {keyword_text} "  # 3x weight for keywords
            
            # Add main content
            combined_text += content
            
            # Tokenize and add to corpus
            tokenized_doc = self._preprocess_text(combined_text, "de")
            self.tokenized_corpus["de"].append(tokenized_doc)
        
        # Create BM25 indexes with improved parameters
        if self.tokenized_corpus["en"]:
            print("Creating English BM25 index...")
            # Adjust BM25 parameters for small documents
            # k1: Controls term frequency saturation (lower value = less saturation)
            # b: Controls document length normalization (lower value = less normalization)
            self.bm25_indexes["en"] = BM25Okapi(self.tokenized_corpus["en"], k1=1.2, b=0.5)
        
        if self.tokenized_corpus["de"]:
            print("Creating German BM25 index...")
            self.bm25_indexes["de"] = BM25Okapi(self.tokenized_corpus["de"], k1=1.2, b=0.5)
        
        print("BM25 indexes built successfully")
    
    def detect_language(self, query: str) -> str:
        """
        Detect the language of the query
        
        Args:
            query: User query
            
        Returns:
            Language code ('en' or 'de')
        """
        try:
            lang = detect(query)
            if lang == 'en':
                return 'en'
            elif lang == 'de':  # Only detect standard German
                return 'de'
            else:
                print(f"Language detected as {lang}, defaulting to English")
                return 'en'
        except:
            print("Language detection failed, defaulting to English")
            return 'en'
    
    def translate_query(self, query: str, source_lang: str, target_lang: str) -> str:
        """
        Translate query from source language to target language
        
        Args:
            query: Query to translate
            source_lang: Source language code ('en' or 'de')
            target_lang: Target language code ('en' or 'de')
            
        Returns:
            Translated query
        """
        try:
            translator = GoogleTranslator(source=source_lang, target=target_lang)
            translated = translator.translate(query)
            return translated
        except Exception as e:
            print(f"Translation error: {e}")
            return query  # Return original query if translation fails
    
    def search(self, query: str, top_k: int = 5) -> List[Dict]:
        """
        Search for documents matching the query in both languages
        
        Args:
            query: User query
            top_k: Number of top results to return
            
        Returns:
            List of retrieved documents with scores
        """
        # Detect query language
        query_lang = self.detect_language(query)
        
        # Get the other language
        other_lang = "de" if query_lang == "en" else "en"
        
        print(f"Query language detected as {query_lang}")
        
        # Translate query to the other language
        translated_query = self.translate_query(query, query_lang, other_lang)
        
        print(f"Original query: {query}")
        print(f"Translated query: {translated_query}")
        
        # Search in original language
        original_results = self._search_language(query, query_lang, top_k)
        
        # Search in translated language
        translated_results = self._search_language(translated_query, other_lang, top_k)
        
        # Combine results
        all_results = original_results + translated_results
        
        # Sort by score (descending)
        all_results.sort(key=lambda x: x["score"], reverse=True)
        
        # Return top_k results
        return all_results[:top_k]
    
    def _search_language(self, query: str, language: str, top_k: int) -> List[Dict]:
        """
        Search for documents in a specific language with improved scoring
        
        Args:
            query: Query (in the language specified)
            language: Language to search in ('en' or 'de')
            top_k: Number of top results to return
            
        Returns:
            List of retrieved documents with scores
        """
        results = []
        
        # Check if we have documents and an index for this language
        if not self.documents[language] or self.bm25_indexes[language] is None:
            return results
        
        # Preprocess query
        tokenized_query = self._preprocess_text(query, language)
        
        if not tokenized_query:
            return results
        
        # Get BM25 scores
        scores = self.bm25_indexes[language].get_scores(tokenized_query)
        
        # Get top_k document indices
        top_indices = np.argsort(scores)[::-1][:top_k]
        
        # Create result objects - include ALL results even with 0 score for small corpora
        for idx in top_indices:
            doc = self.documents[language][idx]
            
            # Create a readable result object
            result = {
                "document": doc,
                "score": float(scores[idx]),
                "language": language,
                "title": doc.get("title", ""),
                "summary": doc.get("summary", ""),
                "content_snippet": self._get_content_snippet(doc.get("main_content", ""), tokenized_query)
            }
            
            results.append(result)
        
        return results
    
    def _get_content_snippet(self, content: str, query_tokens: List[str], max_length: int = 200) -> str:
        """
        Extract a relevant snippet from the content based on query tokens
        
        Args:
            content: Document content
            query_tokens: Tokenized query
            max_length: Maximum snippet length
            
        Returns:
            Content snippet
        """
        if not content or not query_tokens:
            return ""
        
        # Simple approach: Find first occurrence of any query token
        content_lower = content.lower()
        best_pos = len(content)
        
        for token in query_tokens:
            pos = content_lower.find(token)
            if pos != -1 and pos < best_pos:
                best_pos = pos
        
        # If no query tokens found, return the beginning of the content
        if best_pos == len(content):
            best_pos = 0
        
        # Find a good starting position (start of a sentence if possible)
        start = max(0, best_pos - 100)
        while start > 0 and content[start] not in ".!?\n":
            start -= 1
        
        if start > 0:
            start += 1  # Skip the punctuation
        
        # Find a good ending position (end of a sentence if possible)
        end = min(len(content), best_pos + max_length)
        while end < len(content) and content[end] not in ".!?\n":
            end += 1
        
        if end < len(content):
            end += 1  # Include the punctuation
        
        # Extract snippet
        snippet = content[start:end].strip()
        
        # Add ellipsis if needed
        if start > 0:
            snippet = "..." + snippet
        
        if end < len(content):
            snippet = snippet + "..."
        
        return snippet

## 2. Testing with Sample Documents

A small sample of documents to test the implementation.

In [29]:
import tempfile
import shutil

In [30]:
# Create temporary directory
temp_dir = tempfile.mkdtemp()
print(f"Created temporary directory: {temp_dir}")

Created temporary directory: /var/folders/97/_zj78ff90g9cg8jqzm6jx0cr0000gp/T/tmppnt6j7d2


In [31]:
# Sample documents
sample_docs = [
    {
        "language": "en",
        "title": "Artificial Intelligence Research at ETH",
        "date": "2023-01-15",
        "source": "ETH News",
        "main_content": "ETH Zurich is at the forefront of AI research in Europe. Researchers are working on machine learning, computer vision, and natural language processing. The university recently established a new AI center.",
        "summary": "ETH Zurich leads AI research with focus on machine learning and NLP.",
        "named_entities": ["ETH Zurich", "Europe", "AI center"],
        "topics": ["Artificial Intelligence", "Research", "Technology"],
        "keywords": ["AI", "machine learning", "research", "ETH"]
    },
    {
        "language": "en",
        "title": "Sustainable Energy Solutions",
        "date": "2022-11-05",
        "source": "ETH News",
        "main_content": "Researchers at ETH Zurich have developed new solar cell technology with improved efficiency. The cells convert more sunlight into electricity and can be manufactured at a lower cost. This could accelerate the adoption of renewable energy worldwide.",
        "summary": "ETH develops more efficient and cost-effective solar cells.",
        "named_entities": ["ETH Zurich"],
        "topics": ["Renewable Energy", "Sustainability", "Technology"],
        "keywords": ["solar cells", "renewable energy", "sustainability"]
    },
    {
        "language": "de",
        "title": "Künstliche Intelligenz Forschung an der ETH",
        "date": "2023-02-10",
        "source": "ETH Nachrichten",
        "main_content": "Die ETH Zürich ist führend in der KI-Forschung in Europa. Forscher arbeiten an maschinellem Lernen, Computer Vision und Verarbeitung natürlicher Sprache. Die Universität hat kürzlich ein neues KI-Zentrum eingerichtet.",
        "summary": "ETH Zürich führt KI-Forschung mit Fokus auf maschinelles Lernen und NLP.",
        "named_entities": ["ETH Zürich", "Europa", "KI-Zentrum"],
        "topics": ["Künstliche Intelligenz", "Forschung", "Technologie"],
        "keywords": ["KI", "maschinelles Lernen", "Forschung", "ETH"]
    },
    {
        "language": "de",
        "title": "Nachhaltige Energielösungen",
        "date": "2022-12-15",
        "source": "ETH Nachrichten",
        "main_content": "Forscher der ETH Zürich haben eine neue Solarzellentechnologie mit verbesserter Effizienz entwickelt. Die Zellen wandeln mehr Sonnenlicht in Elektrizität um und können zu geringeren Kosten hergestellt werden. Dies könnte die Einführung erneuerbarer Energien weltweit beschleunigen.",
        "summary": "ETH entwickelt effizientere und kostengünstigere Solarzellen.",
        "named_entities": ["ETH Zürich"],
        "topics": ["Erneuerbare Energie", "Nachhaltigkeit", "Technologie"],
        "keywords": ["Solarzellen", "erneuerbare Energie", "Nachhaltigkeit"]
    }
]

In [32]:
# Write sample documents to temporary directory
for i, doc in enumerate(sample_docs):
    file_path = os.path.join(temp_dir, f"doc_{i+1}.json")
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(doc, f, ensure_ascii=False, indent=2)

print(f"Created {len(sample_docs)} sample documents")

Created 4 sample documents


## 3. Initialize and Test the Retriever

In [33]:
import nltk

# Download required NLTK resources
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/donishchuk/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/donishchuk/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [36]:
# Initialize the retriever with our sample documents
retriever = MultilingualBM25Retriever(temp_dir)

Loading documents from /var/folders/97/_zj78ff90g9cg8jqzm6jx0cr0000gp/T/tmppnt6j7d2...
Loaded 2 English documents and 2 German documents
Building BM25 indexes...
Creating English BM25 index...
Creating German BM25 index...
BM25 indexes built successfully


In [37]:
# Test with English query
en_query = "What is ETH doing in artificial intelligence research?"
print("\n===== English Query =====")
results_en = retriever.search(en_query, top_k=3)

print(f"\nResults for query: '{en_query}'")
for i, result in enumerate(results_en):
    print(f"\n{i+1}. [{result['language'].upper()}] {result['title']} (Score: {result['score']:.4f})")
    print(f"Summary: {result['summary']}")
    print(f"Snippet: {result['content_snippet']}")


===== English Query =====
Query language detected as en
Original query: What is ETH doing in artificial intelligence research?
Translated query: Was macht ETH in der künstlichen Intelligenzforschung?

Results for query: 'What is ETH doing in artificial intelligence research?'

1. [DE] Nachhaltige Energielösungen (Score: -0.0377)
Summary: ETH entwickelt effizientere und kostengünstigere Solarzellen.
Snippet: Forscher der ETH Zürich haben eine neue Solarzellentechnologie mit verbesserter Effizienz entwickelt. Die Zellen wandeln mehr Sonnenlicht in Elektrizität um und können zu geringeren Kosten hergestellt werden. Dies könnte die Einführung erneuerbarer Energien weltweit beschleunigen.

2. [DE] Künstliche Intelligenz Forschung an der ETH (Score: -0.0462)
Summary: ETH Zürich führt KI-Forschung mit Fokus auf maschinelles Lernen und NLP.
Snippet: Die ETH Zürich ist führend in der KI-Forschung in Europa. Forscher arbeiten an maschinellem Lernen, Computer Vision und Verarbeitung natürlicher 

In [38]:
# Test with German query
de_query = "Was macht die ETH in der KI-Forschung?"
print("\n===== German Query =====")
results_de = retriever.search(de_query, top_k=3)

print(f"\nResults for query: '{de_query}'")
for i, result in enumerate(results_de):
    print(f"\n{i+1}. [{result['language'].upper()}] {result['title']} (Score: {result['score']:.4f})")
    print(f"Summary: {result['summary']}")
    print(f"Snippet: {result['content_snippet']}")


===== German Query =====
Query language detected as de
Original query: Was macht die ETH in der KI-Forschung?
Translated query: What does ETH do in AI research?

Results for query: 'Was macht die ETH in der KI-Forschung?'

1. [DE] Nachhaltige Energielösungen (Score: -0.0377)
Summary: ETH entwickelt effizientere und kostengünstigere Solarzellen.
Snippet: Forscher der ETH Zürich haben eine neue Solarzellentechnologie mit verbesserter Effizienz entwickelt. Die Zellen wandeln mehr Sonnenlicht in Elektrizität um und können zu geringeren Kosten hergestellt werden. Dies könnte die Einführung erneuerbarer Energien weltweit beschleunigen.

2. [DE] Künstliche Intelligenz Forschung an der ETH (Score: -0.0462)
Summary: ETH Zürich führt KI-Forschung mit Fokus auf maschinelles Lernen und NLP.
Snippet: Die ETH Zürich ist führend in der KI-Forschung in Europa. Forscher arbeiten an maschinellem Lernen, Computer Vision und Verarbeitung natürlicher Sprache. Die Universität hat kürzlich ein neues KI-Zent

## 4. Evaluate the Results

How well our BM25 retriever works across languages?

In [39]:
# Define some evaluation metrics

def print_retrieval_stats(query, results):
    """Print statistics about the retrieval results"""
    
    # Count results by language
    en_count = sum(1 for r in results if r['language'] == 'en')
    de_count = sum(1 for r in results if r['language'] == 'de')
    
    # Get original query language
    query_lang = retriever.detect_language(query)
    
    print(f"Query: '{query}'")
    print(f"Detected language: {query_lang.upper()}")
    print(f"Total results: {len(results)}")
    print(f"English results: {en_count}")
    print(f"German results: {de_count}")
    print(f"Cross-lingual results: {en_count if query_lang == 'de' else de_count}")
    
    # Print top result
    if results:
        top = results[0]
        print("\nTop result:")
        print(f"Title: {top['title']}")
        print(f"Language: {top['language'].upper()}")
        print(f"Score: {top['score']:.4f}")
    
    print("\n" + "-"*50)

In [40]:
# Test a few more queries
test_queries = [
    "solar energy research",
    "Solarenergieforschung",
    "machine learning at ETH",
    "maschinelles Lernen an der ETH",
    "sustainable technologies",
    "nachhaltige Technologien"
]

print("\n===== Evaluation =====")
for query in test_queries:
    results = retriever.search(query, top_k=3)
    print_retrieval_stats(query, results)


===== Evaluation =====
Query language detected as en
Original query: solar energy research
Translated query: Solarenergieforschung
Query: 'solar energy research'
Detected language: EN
Total results: 3
English results: 2
German results: 1
Cross-lingual results: 1

Top result:
Title: Sustainable Energy Solutions
Language: EN
Score: 0.0000

--------------------------------------------------
Query language detected as de
Original query: Solarenergieforschung
Translated query: Solar energy research
Query: 'Solarenergieforschung'
Detected language: DE
Total results: 3
English results: 1
German results: 2
Cross-lingual results: 1

Top result:
Title: Nachhaltige Energielösungen
Language: DE
Score: 0.0000

--------------------------------------------------
Query language detected as en
Original query: machine learning at ETH
Translated query: maschinelles Lernen bei ETH
Query: 'machine learning at ETH'
Detected language: EN
Total results: 3
English results: 1
German results: 2
Cross-lingual re